# Simulating cavity equations to gather synthetic data

The idea is to simulate the behavior of a simple dynamical plant with three mechanical modes only: 100 Hz, 40 Hz and 10 Hz. The resulting synthetic data can then be used to train a KIND filter in order to extract stationary-like data snippets from real measured cavity data.

In [1]:
# --! include root folder into PYTHONPATH --!

import os
import sys

thisdir = os.getcwd()
rootdir = os.path.abspath(os.path.join(thisdir, '..', '..'))
sys.path.append(rootdir)

# --! import Python libraries and KIND framework files --!

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import utils_data
import ex_detuning

### Data simulation

In [2]:
# --! time definitions for simulations --!

timeseries_nsample = 500
timestep = 0.001
t = np.arange(0., timeseries_nsample*timestep, timestep)

In [3]:
# --! obtain stationary detuning data --!

stat_data = []
start_jsample_stat = 0
end_jsample_stat = 500

# --! define stationary mechanical modes that perturb our cavity
mm100_neg = ex_detuning.mechanical_mode(100., 1e3, -0.2, [-1., -1.])
mm40 = ex_detuning.mechanical_mode(40., 400., 0.1, [-1., -1.])
mm10 = ex_detuning.mechanical_mode(10., 1e2, 0.1, [-1., -1.])

stat_data.append((
    ex_detuning.sim_cavity_control(
        t, start_jsample=start_jsample_stat, end_jsample=end_jsample_stat,
        lqr_used=False,
        mm_f=np.array([mm100_neg.f]),
        mm_q=np.array([mm100_neg.q]),
        mm_k=np.array([mm100_neg.k]),
        mm_t=np.array([mm100_neg.t]),
        plotted=False), False))

stat_data.append((
    ex_detuning.sim_cavity_control(
        t, start_jsample=start_jsample_stat, end_jsample=end_jsample_stat,
        lqr_used=False,
        mm_f=np.array([mm40.f]),
        mm_q=np.array([mm40.q]),
        mm_k=np.array([mm40.k]),
        mm_t=np.array([mm40.t]),
        plotted=False), False))

stat_data.append((
    ex_detuning.sim_cavity_control(
        t, start_jsample=start_jsample_stat, end_jsample=end_jsample_stat,
        lqr_used=False,
        mm_f=np.array([mm10.f]),
        mm_q=np.array([mm10.q]),
        mm_k=np.array([mm10.k]),
        mm_t=np.array([mm10.t]),
        plotted=False), False))

stat_data.append((
    ex_detuning.sim_cavity_control(
        t, start_jsample=start_jsample_stat, end_jsample=end_jsample_stat,
        lqr_used=False,
        mm_f=np.array([mm100_neg.f, mm10.f]),
        mm_q=np.array([mm100_neg.q, mm10.q]),
        mm_k=np.array([mm100_neg.k, mm10.k]),
        mm_t=np.array([mm100_neg.t, mm10.t]),
        plotted=False), False))

stat_data.append((
    ex_detuning.sim_cavity_control(
        t, start_jsample=start_jsample_stat, end_jsample=end_jsample_stat,
        lqr_used=False,
        mm_f=np.array([mm100_neg.f, mm40.f]),
        mm_q=np.array([mm100_neg.q, mm40.q]),
        mm_k=np.array([mm100_neg.k, mm40.k]),
        mm_t=np.array([mm100_neg.t, mm40.t]),
        plotted=False), False))

stat_data.append((
    ex_detuning.sim_cavity_control(
        t, start_jsample=start_jsample_stat, end_jsample=end_jsample_stat,
        lqr_used=False,
        mm_f=np.array([mm10.f, mm40.f]),
        mm_q=np.array([mm10.q, mm40.q]),
        mm_k=np.array([mm10.k, mm40.k]),
        mm_t=np.array([mm10.t, mm40.t]),
        plotted=False), False))

stat_data.append((
    ex_detuning.sim_cavity_control(
        t, start_jsample=start_jsample_stat, end_jsample=end_jsample_stat,
        lqr_used=False,
        mm_f=np.array([mm100_neg.f, mm10.f, mm40.f]),
        mm_q=np.array([mm100_neg.q, mm10.q, mm40.q]),
        mm_k=np.array([mm100_neg.k, mm10.k, mm40.k]),
        mm_t=np.array([mm100_neg.t, mm10.t, mm40.t]),
        plotted=False), False))


### Simulated data saving

In [4]:
# --! save simulated data --!

def cat_mask(data):
    """ Concatenates mask dimension to main simulated data dimensions. """
    signal = data[0]
    mask = np.ones((signal.shape[0], signal.shape[1], 1)) if data[1] else np.zeros((signal.shape[0], signal.shape[1], 1))
    return np.concatenate([signal, mask], axis=-1)

datasaved = True

if datasaved:
    savedata = np.concatenate([cat_mask(data) for data in stat_data], axis=0)
    print(f'inf > stationary data saved with a shape {savedata.shape}')
    savedir = '../../data/kalman'
    utils_data.write_datafile(f'{savedir}/detuning_sim_stat', savedata)

inf > stationary data saved with a shape (7, 500, 3)
